# TD1 - ETL des données de fréquentation IDFM

Ce notebook est le support de travail et le livrable technique du TD
([consigne](https://www.m2bigdata.biz/td1/td1.html) et
[sources](https://www.m2bigdata.biz/td1/02-sources.html)).

| Source | Statut |
| --- | --- |
| Validations quotidiennes IDFM | À faire |
| Zones de correspondance | **Exemple guidé ci-dessous** |
| Jours fériés | À faire |
| Météo quotidienne Météo-France | À faire |

## Avant de commencer

Le détail de chaque étape est dans la page
[Avant de commencer](https://www.m2bigdata.biz/td1/03-avant-de-commencer.html)
du support. Assurez-vous d'avoir :

- **demandé votre coupon Google Cloud de 50 $** sur la
  [page de demande](https://gcp.secure.force.com/GCPEDU?cid=DxAtageUlo4UYKtmfn3ZkP9T6pFyBH2LQfElsi%2BJlm0eEoqvjpDE4I%2FSZPgHAvFk/)
  avec votre adresse `@etudiant.univ-reims.fr`
  ([slides d'aide](https://docs.google.com/presentation/d/1LMB4PFOYiZGew0tk_D6E6iEKuaveP_6ED_jOBB6r8aU/edit?slide=id.g1bcc591acc_0_13#slide=id.g1bcc591acc_0_13)) ;
- **créé un projet** dans la
  [console Google Cloud](https://console.cloud.google.com/), lié au compte de
  facturation crédité ;
- **enregistré une copie** de ce notebook dans votre Drive
  (**Fichier -> Enregistrer une copie dans Drive**).

Puis collez l'**ID du projet** (pas son nom) dans la variable `PROJET` de la
cellule **Configuration**, et exécutez les cellules **dans l'ordre**.

## 0.1 - Configuration

In [ ]:
%pip -q install google-cloud-bigquery google-cloud-storage db-dtypes pyarrow pandas requests

In [ ]:
import hashlib
from datetime import datetime, timezone
from zipfile import ZipFile
import pandas as pd

import requests
from google.api_core.exceptions import NotFound, PreconditionFailed, Forbidden
from google.cloud import bigquery, storage
from google.colab import auth
auth.authenticate_user()

# IMPORTANT : Remplacez par l'ID de votre projet GCP actif avec facturation activée
PROJET = "kbsf-391012"
LOCALISATION = "EU"
BUCKET = f"{PROJET}-td1-idfm"
DATASET_SILVER = "silver_idfm"
DATASET_GOLD = "gold_idfm"

try:
    client = bigquery.Client(project=PROJET, location=LOCALISATION)
    client.default_query_job_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=1 * 1024**3  # 1 Gio facturé au maximum par requête
    )
    gcs = storage.Client(project=PROJET)

    # Tentative de création/récupération du bucket GCS
    try:
        bucket = gcs.get_bucket(BUCKET)
    except NotFound:
        bucket = gcs.create_bucket(BUCKET, location=LOCALISATION)

    # Création des datasets BigQuery
    for dataset_id in (DATASET_SILVER, DATASET_GOLD):
        dataset = bigquery.Dataset(f"{PROJET}.{dataset_id}")
        dataset.location = LOCALISATION
        client.create_dataset(dataset, exists_ok=True)

    print(f"🎉 Configuration réussie !")
    print(f"Bronze : gs://{BUCKET}")
    print(f"Silver : {PROJET}.{DATASET_SILVER}")
    print(f"Gold   : {PROJET}.{DATASET_GOLD}")

except Forbidden as e:
    print("❌ Erreur d'accès GCP (403 Forbidden).")
    print("Veuillez vérifier que :")
    print(f"  1. L'ID du projet \"{PROJET}\" est correct.")
    print("  2. Un compte de facturation actif (ex: votre coupon de 50$) est bien lié à ce projet dans la console Google Cloud.")
    print(f"Détail de l'erreur : {e}")
except Exception as e:
    print(f"❌ Une erreur inattendue est survenue : {e}")

❌ Erreur d'accès GCP (403 Forbidden).
Veuillez vérifier que :
  1. L'ID du projet "kbsf-391012" est correct.
  2. Un compte de facturation actif (ex: votre coupon de 50$) est bien lié à ce projet dans la console Google Cloud.
Détail de l'erreur : 403 POST https://storage.googleapis.com/storage/v1/b?project=kbsf-391012&prettyPrint=false: The billing account for the owning project is disabled in state absent


### Documentation

| Référence | Pour quoi faire |
| --- | --- |
| [Fonctions et opérateurs GoogleSQL](https://cloud.google.com/bigquery/docs/reference/standard-sql/functions-and-operators) | `CAST`, `TRIM`, `NULLIF`, `COUNTIF`... |
| [DDL : CREATE TABLE](https://cloud.google.com/bigquery/docs/reference/standard-sql/data-definition-language) | Schéma explicite, `OPTIONS(description=...)` |
| [Client Python BigQuery](https://cloud.google.com/python/docs/reference/bigquery/latest/google.cloud.bigquery.client.Client) | `client.query`, `client.load_table_from_uri` |
| [Client Python Cloud Storage](https://cloud.google.com/python/docs/reference/storage/latest) | `bucket.blob(...)`, `upload_from_string` |

## 0.2 - Exemple : les zones de correspondance

Une zone regroupe des arrêts proches entre lesquels on change à pied. On suit
le même chemin pour chaque source : **API -> Bronze -> Silver**.

### Source de l'exemple

On conserve le fichier Parquet de l'API tel quel en Bronze. Parquet embarque
les noms et les types des colonnes.

In [ ]:
URL_ZONES = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/zones-de-correspondance/exports/parquet"

### Bronze : collecter une fois, puis réutiliser

Le chemin contient la date du snapshot. Si le fichier existe, on le réutilise.
Pour actualiser la source, choisir un nouveau snapshot et conserver le précédent.
L'URL, les paramètres, la date de collecte et l'empreinte SHA-256 sont enregistrés
avec le fichier.

In [ ]:
SNAPSHOT = "2026-10-09"  # À fixer au premier lancement
CHEMIN_ZONES = f"bronze/idfm/zones_de_correspondance/snapshot={SNAPSHOT}/zones-de-correspondance.parquet"
blob = bucket.blob(CHEMIN_ZONES)

if blob.exists():
    print("Snapshot existant réutilisé :", f"gs://{BUCKET}/{CHEMIN_ZONES}")
else:
    reponse = requests.get(URL_ZONES, params={"limit": -1}, timeout=120)
    reponse.raise_for_status()
    blob.metadata = {
        "url_source": URL_ZONES,
        "parametres": "limit=-1",
        "collecte_utc": datetime.now(timezone.utc).isoformat(),
        "sha256": hashlib.sha256(reponse.content).hexdigest(),
    }
    try:
        # La condition empêche aussi l'écrasement si un autre lancement a créé le fichier.
        blob.upload_from_string(
            reponse.content, content_type="application/parquet", if_generation_match=0
        )
        print("Snapshot créé :", f"gs://{BUCKET}/{CHEMIN_ZONES}")
    except PreconditionFailed:
        print("Snapshot créé par un autre lancement : réutilisation.")

Snapshot créé : gs://td1-41374-td1-idfm/bronze/idfm/zones_de_correspondance/snapshot=2026-10-09/zones-de-correspondance.parquet


### Silver : charger et transformer en SQL

1. Charger le Parquet dans une table de chargement avec son schéma embarqué.
2. Créer la table Silver avec un schéma explicite et l'alimenter en SQL.
3. Afficher les contrôles et examiner les résultats.

À chaque relance, les tables sont remplacées pour éviter d'accumuler des doublons.


In [ ]:
TABLE_CHARGEMENT = f"{PROJET}.{DATASET_SILVER}.zone_correspondance_chargement"

config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition="WRITE_TRUNCATE",  # relancer remplace la table : pas de doublon
)
client.load_table_from_uri(f"gs://{BUCKET}/{CHEMIN_ZONES}", TABLE_CHARGEMENT, job_config=config).result()

client.query(f"SELECT * FROM `{TABLE_CHARGEMENT}` LIMIT 5").to_dataframe()

,zdcid,zdcversion,zdccreated,zdcchanged,zdcname,zdcxepsg2154,zdcyepsg2154,zdctown,zdcpostalregion,zdctype
0,411379,1088971,2015-01-26 00:00:00+00:00,2019-04-24 10:42:33+00:00,Château-Thierry,729938,6882094,Château-Thierry,02168,railStation
1,411382,1090890,2015-01-26 00:00:00+00:00,2023-03-03 10:59:23+00:00,Chézy-sur-Marne,726734,6876797,Chézy-sur-Marne,02186,railStation
2,411373,1095721,2015-01-26 00:00:00+00:00,2026-02-09 14:58:48+00:00,La Ferté-Milon,708818,6897761,La Ferté-Milon,02307,railStation
3,479607,1203997,2020-07-21 16:05:29+00:00,2022-08-03 17:24:24+00:00,Pharmacie,713969,6880142,Montreuil-aux-Lions,02521,onstreetBus
4,411385,1119782,2015-01-26 00:00:00+00:00,2023-03-03 14:05:08+00:00,Nogent-L'Artaud - Charly,723611,6874366,Nogent-l'Artaud,02555,railStation


In [ ]:
TABLE_ZONES = f"{PROJET}.{DATASET_SILVER}.zone_correspondance"

client.query(f"""
    CREATE OR REPLACE TABLE `{TABLE_ZONES}` (
      zdc_id INT64 NOT NULL OPTIONS(description="Identifiant de la zone de correspondance. Clé unique."),
      nom_zone STRING NOT NULL OPTIONS(description="Nom de la zone."),
      commune STRING OPTIONS(description="Commune de la zone. NULL si non renseignée."),
      code_insee_commune STRING NOT NULL OPTIONS(description="Code INSEE de la commune, sur 5 caractères."),
      type_zone STRING OPTIONS(description="Mode de transport principal (ex. metroStation). NULL si non renseigné."),
      x_lambert93 FLOAT64 NOT NULL OPTIONS(description="Coordonnée X du centre de la zone, en mètres (Lambert 93)."),
      y_lambert93 FLOAT64 NOT NULL OPTIONS(description="Coordonnée Y du centre de la zone, en mètres (Lambert 93).")
    );

    INSERT INTO `{TABLE_ZONES}`
    SELECT
      SAFE_CAST(zdcid AS INT64),
      NULLIF(TRIM(zdcname), ''),
      NULLIF(TRIM(zdctown), ''),
      NULLIF(TRIM(zdcpostalregion), ''),
      NULLIF(TRIM(zdctype), ''),
      SAFE_CAST(zdcxepsg2154 AS FLOAT64),
      SAFE_CAST(zdcyepsg2154 AS FLOAT64)
    FROM `{TABLE_CHARGEMENT}`;
""").result()

print(client.get_table(TABLE_ZONES).num_rows, "lignes Silver")

15597 lignes Silver


### Examiner les contrôles


In [ ]:
bilan = client.query(f"""
    SELECT
      (SELECT COUNT(*) FROM `{TABLE_CHARGEMENT}`) AS nb_lignes_source,
      COUNT(*) AS nb_lignes_silver,
      COUNT(DISTINCT zdc_id) AS nb_identifiants,
      COUNT(*) - COUNT(DISTINCT zdc_id) AS nb_identifiants_en_double
    FROM `{TABLE_ZONES}`
""").to_dataframe()
display(bilan)

valeurs_nulles = client.query(f"""
    SELECT
      COUNTIF(zdc_id IS NULL) AS zdc_id,
      COUNTIF(nom_zone IS NULL) AS nom_zone,
      COUNTIF(commune IS NULL) AS commune,
      COUNTIF(code_insee_commune IS NULL) AS code_insee_commune,
      COUNTIF(type_zone IS NULL) AS type_zone,
      COUNTIF(x_lambert93 IS NULL) AS x_lambert93,
      COUNTIF(y_lambert93 IS NULL) AS y_lambert93
    FROM `{TABLE_ZONES}`
""").to_dataframe()

# Une ligne par colonne, avec son nom et son nombre de valeurs nulles.
valeurs_nulles = valeurs_nulles.T
valeurs_nulles.index.name = "colonne"
valeurs_nulles.columns = ["nb_valeurs_nulles"]
display(valeurs_nulles)


,nb_lignes_source,nb_lignes_silver,nb_identifiants,nb_identifiants_en_double
0,15597,15597,15597,0


,nb_valeurs_nulles
colonne,
zdc_id,0
nom_zone,0
commune,0
code_insee_commune,0
type_zone,1
x_lambert93,0
y_lambert93,0


In [ ]:
client.query(f"SELECT * FROM `{TABLE_ZONES}` LIMIT 5").to_dataframe()

,zdc_id,nom_zone,commune,code_insee_commune,type_zone,x_lambert93,y_lambert93
0,60745,Route de Grigny,Ris-Orangis,91521,None,656092.0,6839106.0
1,69636,La Végétale,Limeil-Brévannes,94044,liftStation,661707.0,6849200.0
2,74103,Limeil-Brévannes,Valenton,94074,liftStation,661187.0,6850351.0
3,69685,Valenton,Valenton,94074,liftStation,661254.0,6849974.0
4,69591,Villa Nova,Villeneuve-Saint-Georges,94078,liftStation,660639.0,6848466.0


## À vous

Le TD porte sur **quatre sources** ; les zones de correspondance sont
l'exemple ci-dessus. Reproduisez ce chemin (**API -> Bronze -> Silver**)
pour les trois autres : validations, jours fériés et météo.

La couche **Gold** se définit ensuite, une fois toutes les sources en Silver :
elle demande en général des jointures entre plusieurs tables. Construisez-la,
vérifiez-la et proposez les exemples d'exploitation demandés dans la
[consigne](https://www.m2bigdata.biz/td1/td1.html).
